In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from src.preprocessing import read_flow
df_flow, df_list, df_wsp = read_flow(
    '11-Mar-2026 Treg .wsp', 'Treg FCS', 'T Cells')

In [ ]:
df_flow.drop(columns=['FSC-A FSC - Area', 'FSC-H FSC - Height',
                      'FSC-W FSC - Width', 'SSC-A SSC - Area', 'SSC-H SSC - Height',
                      'SSC-W SSC - Width', 'AIM Dump', 'L/D', 'TIME Time Stamp',  '[AF color 1]-A [AF color 1] - Area', 'CD45', 'CD3'], inplace=True)

In [ ]:
df_flow_counts = df_flow.select_dtypes(include=[np.number])

In [ ]:
from src.preprocessing import pd_to_adata
adata = pd_to_adata(df_flow, df_flow_counts)

In [ ]:
from run_pipeline import clustering_pipeline
adata = clustering_pipeline(adata)

In [ ]:
from run_pipeline import dem_ranked
adata, unique_values = dem_ranked(adata)

In [ ]:
cluster_annotation = {'celltype': []}
cluster_to_genes = {
    '0': 'HLA-DR+ CD45RA+CCR7+ (0)',
    '2': 'HLA-DR+ CD45RA-CCR7- (2)',
    '7': 'CD4+ TEMRA (7)',
    '3': 'CD8+ EM CD103+CD69+ (3)',
    '6': 'CD8+ TEMRA (6)',
    '5': 'CD4+ TCM CD69+ (5)',
    '4': 'CD4+ TCM (4)',
    '1': 'CD4+ EM (1)',
    '8': 'CD4+ EM CD103+CD69+ (8)'

}
cluster_annotation['celltype'] = [cluster_to_genes[leiden]
                                  for leiden in adata.obs['leiden']]
adata.obs["leiden_annotation"] = cluster_annotation['celltype']

print(adata.obs[["leiden", "leiden_annotation"]].head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.umap(
    adata,
    color=['leiden_annotation'],
    cmap='turbo',
    title='CyTOF Annotated Leiden Clusters',
    show=False,
)

ax = plt.gca()
for cluster in adata.obs['leiden'].cat.categories:
    cluster_mask = adata.obs['leiden'] == cluster
    cluster_coords = adata.obsm['X_umap'][cluster_mask]
    x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
    ax.text(x, y, cluster, color='black', fontsize=10,
            weight='bold', ha='center', va='center')

plt.show()

In [ ]:
for tissue in list(adata.obs.tissue.unique()):
    adata_group = adata[adata.obs['tissue'] == tissue]

    sc.pl.umap(
        adata_group,
        color=['leiden_annotation'],
        title=f'{tissue} Clusters',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for group in list(adata.obs.group.unique()):
    adata_group = adata[adata.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['leiden_annotation'],
        title=f'{group} Clusters',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()